In [295]:
%pip install datasets requests pandas

Note: you may need to restart the kernel to use updated packages.


DEPRECATION: Loading egg at c:\users\admin\anaconda3\lib\site-packages\huggingface_hub-0.29.2-py3.8.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [296]:
import json
import time
import urllib.request
import urllib.error
from datasets import load_dataset
import requests
import pandas as pd

In [297]:
#Check that Ollama is running and available before using the Qwen model, must be installed manually
OLLAMA_URL = "http://localhost:11434"

try:
    response = requests.get(OLLAMA_URL, timeout=5)
    print("Ollama is running.")
except requests.RequestException:
    print("Ollama is not running. Please install and start Ollama.")

Ollama is running.


In [298]:
MODEL = "qwen2.5-coder:7b"

In [299]:
def model(code,desc):

    prompt = f"""
    You are a Python code review and debugging assistant.
    
    Find the errors in this Python code and correct them. Implement test cases when the description is available to make sure the output is correct. Do not claim that tests pass unless they have actually been executed.
    Count all the errors you fixed including repeated fixes and indented changes, and return your answer with no extra text or explanation using exactly this format:

    Corrected code:
    [Complete code after correcting]

    Total Number of errors: [integer]

    
    code:
    {code}
    
    description:
    {desc}

    
    """
    
    request_data = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0}
    }
    
    request = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(request_data).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    start_time = time.perf_counter()
    
    try:
        with urllib.request.urlopen(request, timeout=300) as response:
            result = json.loads(response.read().decode("utf-8"))
    except Exception as e:
        print(f"Model request failed: {e}")
        return None
    
    elapsed_time = time.perf_counter() - start_time
    print(f"\nResponse time: {elapsed_time:.2f} seconds")

    return result["response"]

In [300]:
#dataset for syntax or runtime errors
dataset_syntax = load_dataset("Harryxun/BugsInPy_data",split="train[:30]")
code_syntax = dataset_syntax["buggy"]
sol_syntax = dataset_syntax["fixed"] #for evaluation

In [301]:
#ddataset for logical errors
dataset_logical = load_dataset("Muennighoff/quixbugs",split="train[:30]")
code_logical = dataset_logical["buggy_program"]
desc_logical = dataset_logical["docstring"] #code description
test_logical = dataset_logical["tests"] #for evaluation

In [302]:
#example using code with syntax errors
output_syntax = model(code_syntax[6],"")
print(output_syntax)


Response time: 8.27 seconds
Corrected code:
```python
class SeriesGroupBy(GroupBy):
    res = np.zeros(len(ri), dtype=out.dtype)
    res[ids[idx]] = out
    return Series(res, index=ri, name=self._selection_name)

@Appender(Series.describe.__doc__)
def describe(self, **kwargs):
    return Series.describe(self, **kwargs)
```

Total Number of errors: 2


In [303]:
#example using code with logical errors
output_logical = model(code_logical[7],desc_logical[7])
print(output_logical)


Response time: 3.38 seconds
Corrected code:
```python
def flatten(arr):
    for x in arr:
        if isinstance(x, list):
            for y in flatten(x):
                yield y
        else:
            yield x
```

Total Number of errors: 1


In [304]:
#My own example - simple
code_simple = """
def sub(a,b):
-x = a
    y = b
return -x + y
"""
desc_simple = "A code to subtract two numbers"
output_simple = model(code_simple,desc_simple)
print(output_simple)


Response time: 2.94 seconds
Corrected code:
```python
def sub(a, b):
    x = a
    y = b
    return x - y
```

Total Number of errors: 2


In [305]:
#generated example by ChatGPT - Asked to come up with a hard example
code_hard = """
def filter_numbers(numbers):
    result = list()

    for n in numbers:
        if n % 2 == 0 or n > 0:
            if n not in result:
                result.append(n)

    for i in range(len(result)):
        if result[i] < result[i - 1]:
            result[i], result[i - 1] = result[i - 1], result[i]

    return result
"""
desc_hard = "Write a function that returns a sorted list of unique even numbers greater than zero. Return an empty list if no numbers qualify"
output_hard = model(code_hard,desc_hard)
print(output_hard)


Response time: 4.54 seconds
Corrected code:
```python
def filter_numbers(numbers):
    result = list()

    for n in numbers:
        if n % 2 == 0 and n > 0:
            if n not in result:
                result.append(n)

    for i in range(1, len(result)):
        if result[i] < result[i - 1]:
            result[i], result[i - 1] = result[i - 1], result[i]

    return result
```

Total Number of errors: 1


In [ ]:
#try the model

# Enter Python code for the model to review
user_code = input("Paste your Python code: ")

# Describe what the code is supposed to do
user_desc = input("Describe what the code should do: ")

user_example = model(user_code, user_desc)

print("\nModel Output:")
print(user_example)